In [ ]:
# Clean uninstall (ignore errors if not installed)
!pip uninstall -y langchain langchain-community langchain-core faiss-cpu sentence-transformers openai

# Install compatible versions (VERY IMPORTANT)
!pip install langchain==0.1.16 \
             langchain-community==0.0.32 \
             langchain-core==0.1.42 \
             faiss-cpu \
             sentence-transformers \
             openai==0.28.1

Found existing installation: langchain 1.2.12
Uninstalling langchain-1.2.12:
  Successfully uninstalled langchain-1.2.12
Found existing installation: langchain-core 1.2.19
Uninstalling langchain-core-1.2.19:
  Successfully uninstalled langchain-core-1.2.19
Found existing installation: sentence-transformers 5.3.0
Uninstalling sentence-transformers-5.3.0:
  Successfully uninstalled sentence-transformers-5.3.0
Found existing installation: openai 2.28.0
Uninstalling openai-2.28.0:
  Successfully uninstalled openai-2.28.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 817.7/817.7 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 91.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.5/287.5 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.6 MB/s eta 0:00:00


In [ ]:
# Import text splitter to break large text into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# FAISS vector database
from langchain_community.vectorstores import FAISS

# Embedding model (converts text → vectors)
from langchain_community.embeddings import HuggingFaceEmbeddings

# RAG pipeline (Retrieval + LLM)
from langchain.chains import RetrievalQA

# OpenAI LLM
from langchain_community.llms import OpenAI

import os

In [ ]:
# Set your OpenAI API key here
# Required for accessing the LLM (GPT model)

os.environ["OPENAI_API_KEY"] = ""

In [ ]:
# This is our dataset (documents)
# In real-world, this can be PDFs, websites, or databases

documents = [
    "Machine Learning is a subset of AI that focuses on learning from data.",
    "Deep Learning uses neural networks with many layers.",
    "RAG stands for Retrieval Augmented Generation.",
    "LangChain helps in building LLM-powered applications.",
    "FAISS is used for efficient similarity search and vector storage.",
    "Embeddings convert text into numerical vectors for semantic understanding.",
    "Prompt engineering is important for improving LLM outputs."
]

In [ ]:
# Split large text into smaller chunks
# Why? → LLMs and embeddings work better on smaller pieces of text

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,     # max size of each chunk
    chunk_overlap=20    # overlap helps maintain context between chunks
)

# Convert raw text into document chunks
docs = text_splitter.create_documents(documents)

In [ ]:
# Load embedding model (HuggingFace)
# Converts text → numerical vectors

embeddings = HuggingFaceEmbeddings()

# Store embeddings in FAISS vector database
# FAISS allows fast similarity search

vectorstore = FAISS.from_documents(docs, embeddings)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
# Retriever fetches relevant documents based on user query

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 2}  # return top 2 most relevant chunks
)

In [ ]:
# Combine retrieval + LLM to form RAG system

qa_chain = RetrievalQA.from_chain_type(
    llm=OpenAI(),        # language model (GPT)
    retriever=retriever, # retrieves relevant context
    chain_type="stuff"   # method to combine retrieved docs
)

/usr/local/lib/python3.12/dist-packages/langchain_core/_api/deprecation.py:117: LangChainDeprecationWarning: The class `langchain_community.llms.openai.OpenAI` was deprecated in langchain-community 0.0.10 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAI`.
  warn_deprecated(


In [ ]:
# Ask one question at a time (works perfectly in Colab)

query = input("Ask a question: ")

response = qa_chain.run(query)

print("🤖 Answer:", response)